# Fundamentos de Ingeniería de Datos
## Extracción: de la capa RAW a la capa EXTRACTED

```
INGESTA (ya hecho)                                  EXTRACCIÓN (hoy)
Kaggle → CSV → SQLite: raw_vg_engagement         →  extracted_raw_vg_engagement
                       raw_vg_global_codificado  →  extracted_vg_global_codificado
                       raw_vg_ventas_ratings     →  extracted_vg_ventas_ratings
                                                    + control de cambios (un cdc_control_<tabla>.json por tabla)
                                                    + log de auditoría   (extraccion_log.jsonl, compartido)
```

**Tema 2.1 — Extracción de datos:** formatos y control de cambios.

### ¿Qué es la capa EXTRACTED?

Son **los mismos datos de `raw_<etiqueta>`**, sin limpiar todavía, pero con **4 columnas de metadatos** que dicen cuándo, de dónde y en qué lote llegó cada fila.

| Capa | Tabla | Qué contiene |
|---|---|---|
| RAW | `raw_<etiqueta>` | Datos originales, tal como llegaron (no se modifica) |
| **EXTRACTED** | **`extracted_<etiqueta>`** | **Los mismos datos + metadatos de extracción** |
| TRANSFORMED | `transformed_<etiqueta>` | Datos limpios (siguiente notebook) |

> **Regla de oro:** cada capa lee solo de la anterior. `raw_<etiqueta>` se lee en **modo solo lectura**.

### Organización del notebook

| Parte | Qué contiene |
|---|---|
| **Parte común** | Configuración, funciones de huella, control de cambios, log y `extraer()`: se definen **una sola vez** |
| **Módulo 1** | `raw_vg_engagement` → `extracted_raw_vg_engagement` |
| **Módulo 2** | `raw_vg_global_codificado` → `extracted_vg_global_codificado` |
| **Módulo 3** | `raw_vg_ventas_ratings` → `extracted_vg_ventas_ratings` |
| **Cierre** | Log de operaciones de los 3 módulos y resumen |

Cada módulo empieza definiendo **sus** tablas (`TABLA_RAW`, `TABLA_EXTRACTED`) y **su** archivo de control (`CDC_PATH`). Las funciones de la parte común usan esas variables, así que trabajan sobre el dataset del módulo que se está ejecutando.

### Configuración del entorno

In [ ]:
# ── Importar utils.py ──────────────────────────────────────────────────────

import sys
import os
sys.path.append(os.path.abspath('../Utilidades'))   # el notebook se ejecuta desde Codigos/

from utils import PROYECTO_BASE, DB_PATH, inspeccionar_dataset, listar_tablas

print()
print('utils.py importado correctamente')
print(f'Base de datos: {DB_PATH}')
print()

In [ ]:
# ── Librerías y rutas comunes ──────────────────────────────────────────────

import os
import json
import hashlib
import sqlite3
import pandas as pd
from datetime import datetime




# ----------- Carpetas de esta capa -----------

EXTRACCION_PATH = f'{PROYECTO_BASE}/Extraccion_de_Datos'
LOGS_PATH       = f'{EXTRACCION_PATH}/Logs'
EXPORTS_PATH    = f'{EXTRACCION_PATH}/Exportaciones'

# ----------- Archivos -----------

LOG_PATH = f'{LOGS_PATH}/extraccion_log.jsonl'    # historial de operaciones (compartido por los 3 módulos)

# Las tablas (TABLA_RAW, TABLA_EXTRACTED) y el control de cambios (CDC_PATH)
# se definen al inicio de cada módulo.


# -----------  Si no existen, crearlos -----------
os.makedirs(LOGS_PATH, exist_ok=True)
os.makedirs(EXPORTS_PATH, exist_ok=True)




print()
print(f'Logs        : {LOGS_PATH}')
print(f'Exportación : {EXPORTS_PATH}')
print()

---
## PARTE COMÚN — Revisar la base de datos

Antes de extraer, verificamos qué tablas existen en el `.db`.

In [ ]:
# ── ¿Qué tablas hay en el .db? ─────────────────────────────────────────────

listar_tablas(DB_PATH);


In [ ]:
# ── Eliminar una tabla ─────────────────────────────────────────────────────

TABLA_A_BORRAR = 'extracted_df1'      # cambia el nombre de la tabla

conn = sqlite3.connect(DB_PATH)       # conexión normal (NO en modo solo lectura)
conn.execute(f'DROP TABLE IF EXISTS {TABLA_A_BORRAR}')
conn.commit()
conn.close()

print(f"Tabla '{TABLA_A_BORRAR}' eliminada")

listar_tablas(DB_PATH);

---
## PARTE COMÚN — Metadatos de extracción

A cada fila le agregamos 4 columnas. Empiezan con guion bajo `_` para distinguirlas de las columnas originales.

| Columna | Contenido | ¿Para qué sirve? |
|---|---|---|
| `_batch_id` | Número de lote: 1 la primera vez, 2 la siguiente… | Saber en qué ejecución llegó cada fila |
| `_extracted_at` | Fecha y hora de la extracción | Medir la **frescura** de los datos |
| `_source_table` | Tabla de origen (`raw_<etiqueta>`) | **Lineage**: de dónde viene cada fila |
| `_row_hash` | "Huella digital" del contenido de la fila | **Control de cambios** (Paso 3) |

### La huella (`_row_hash`)

Una **función hash** convierte un texto en un código corto. Si el texto cambia, aunque sea un carácter, el código cambia por completo. Juntamos los valores de la fila en un texto y le calculamos su hash: **misma fila → misma huella; fila modificada → huella nueva**.

In [ ]:
# ── ¿Cómo funciona un hash? ────────────────────────────────────────────────
print()

for texto in ['Wii Sports|Wii|2006.0',
              'Wii Sports|Wii|2007.0',
              'Wii Sports|Wii|2006.0']:

    huella = hashlib.sha256(texto.encode('utf-8')).hexdigest()[:16]
    print(f'{texto:<25} →  {huella}')

print()

In [ ]:
# ── Función para calcular la huella de cada fila ───────────────────────────

def calcular_huella(df):

    '''Regresa la huella
    (16 caracteres) de cada
    fila del DataFrame.'''


    texto_fila = df.map(str).agg('|'.join, axis=1)              # 'v1|v2|v3|...'
    return texto_fila.map(lambda t: hashlib.sha256(t.encode('utf-8')).hexdigest()[:16])



---
## PARTE COMÚN — Control de cambios (CDC)

Este pipeline no se ejecuta una sola vez: se vuelve a ejecutar cada vez que hay una nueva ingesta (según la **frescura** de tu miniespecificación). El **control de cambios** evita copiar todo otra vez:

| Tipo de carga | Qué hace |
|---|---|
| **Completa** (*full load*) | Copia toda la tabla. Ocurre la **primera vez**. |
| **Incremental** | Copia **solo las filas nuevas o modificadas**. Ocurre en las siguientes ejecuciones. |

**¿Cómo sabemos qué es nuevo?** Comparamos la huella de cada fila de `raw_<etiqueta>` con las huellas que **ya están** en `extracted_<etiqueta>`. Solo pasan las que no estaban.

> Usamos huellas porque `raw_<etiqueta>` no tiene una columna de fecha de registro. Si la tuviera, podríamos usar `WHERE fecha > última_fecha_extraída` (estrategia de **marca de agua**).

Cada tabla guarda su estado en **su propio** archivo `cdc_control_<tabla>.json` (último lote, fecha, filas) y cada operación en el log.

In [ ]:
# ── Funciones de control de cambios y log ──────────────────────────────────

# ── Leer cdc ──────────────────────────────────
def leer_cdc():

    '''Lee el estado del control de cambios (cdc_control.json)
       vacío si es la primera vez '''


    if not os.path.exists(CDC_PATH):
        return {}
    with open(CDC_PATH, encoding='utf-8') as f:
        return json.load(f)


# ── Guardar cdc ──────────────────────────────────

def guardar_cdc(estado):
    with open(CDC_PATH, 'w', encoding='utf-8') as f:
        json.dump(estado, f, indent=2, ensure_ascii=False)



# ── Escribe cdc ──────────────────────────────────

def escribir_log(filas_leidas, filas_nuevas, tipo, estado='exitoso', error=None):

    '''Agrega UNA línea al log (formato JSON Lines: un JSON por línea).'''

    evento = {
        'timestamp'    : datetime.now().isoformat(timespec='seconds'),
        'tabla_origen' : TABLA_RAW,
        'tabla_destino': TABLA_EXTRACTED,
        'filas_leidas' : filas_leidas,
        'filas_nuevas' : filas_nuevas,
        'tipo'         : tipo,
        'estado'       : estado,
        'error'        : error,
    }

    with open(LOG_PATH, 'a', encoding='utf-8') as f:     # 'a' = agregar al final
        f.write(json.dumps(evento, ensure_ascii=False) + '\n')






# ── Historial ──────────────────────────────────

def ver_log():
    if not os.path.exists(LOG_PATH):
        print('Todavía no hay operaciones registradas.')
        return pd.DataFrame()
    return pd.read_json(LOG_PATH, lines=True)




print()
print('Funciones listas')
print()

In [ ]:
# ── La función de extracción ───────────────────────────────────────────────

def extraer():

    '''Extrae raw_<etiqueta> → extracted_<etiqueta>, agregando solo las filas nuevas.'''


    try:

        # 1. Leer el origen (solo lectura)
        conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
        df = pd.read_sql_query(f'SELECT * FROM {TABLA_RAW}', conn)



        # 2. Huellas que ya están en EXTRACTED (si la tabla existe)

        existe = conn.execute(
            "SELECT COUNT(*) FROM sqlite_master WHERE type = 'table' AND name = ?",
            (TABLA_EXTRACTED,)
        ).fetchone()[0] > 0


        if existe:
            huellas_previas = pd.read_sql_query(f'SELECT _row_hash FROM {TABLA_EXTRACTED}', conn)['_row_hash']
        else:
            huellas_previas = pd.Series(dtype=str)
        previas = set(huellas_previas)                         # huellas distintas (para comparar)
        filas_previas = len(huellas_previas)                   # filas que ya hay (para contar)
        conn.close()



        # 3. Calcular la huella de cada fila y quedarse solo con las nuevas

        df['_row_hash'] = calcular_huella(df)
        nuevas = df[~df['_row_hash'].isin(previas)].copy()

        tipo = 'full_load' if not previas else 'incremental'
        cdc = leer_cdc()




        # 4. Si hay filas nuevas: metadatos + AGREGAR (append) a extracted_<etiqueta>

        if len(nuevas) > 0:

            batch_id = cdc.get('ultimo_batch', 0) + 1

            nuevas['_batch_id']     = batch_id
            nuevas['_extracted_at'] = datetime.now().isoformat(timespec='seconds')
            nuevas['_source_table'] = TABLA_RAW


            # Orden: columnas originales y al final los metadatos
            columnas = [c for c in nuevas.columns if not c.startswith('_')]
            nuevas = nuevas[columnas + ['_batch_id', '_extracted_at', '_source_table', '_row_hash']]

            conn = sqlite3.connect(DB_PATH)                      # lectura y escritura
            nuevas.to_sql(TABLA_EXTRACTED, conn, if_exists='append', index=False)
            conn.commit()
            conn.close()

            cdc['ultimo_batch'] = batch_id



        # 5. Actualizar control de cambios y log

        cdc.update({

            'tabla_origen'     : TABLA_RAW,
            'tabla_destino'    : TABLA_EXTRACTED,
            'ultima_extraccion': datetime.now().isoformat(timespec='seconds'),
            'filas_leidas'     : len(df),
            'filas_nuevas'     : len(nuevas),
            'filas_acumuladas' : filas_previas + len(nuevas),
            'tipo'             : tipo,
        })


        guardar_cdc(cdc)



        escribir_log(len(df), len(nuevas), tipo)

        print()
        print(f'{TABLA_RAW} → {TABLA_EXTRACTED}  ({tipo})')
        print(f'  Filas leídas : {len(df):,}')
        print(f'  Filas nuevas : {len(nuevas):,}')
        print()
        return nuevas



    except Exception as e:
        escribir_log(0, 0, 'error', estado='fallido', error=str(e))
        raise          # muestra el error en pantalla


print()
print('extraer() definida')
print()

> **¿Por qué `if_exists='append'` y no `'replace'`?** `replace` borraría la tabla y la volvería a crear: perderíamos los lotes anteriores. Con `append` las filas nuevas se **agregan al final** y, gracias al control de cambios, nunca se duplican.

---
## Cómo leer la extracción de cada módulo

En cada módulo, **ejecuta la celda de extracción dos veces** y compara:

- **1ª vez:** carga completa (`full_load`): todas las filas son nuevas.
- **2ª vez:** carga incremental (`incremental`): `raw_<etiqueta>` no cambió, así que **0 filas nuevas**.

---
## Formatos: CSV vs. JSON vs. Parquet

Muchas fuentes entregan **archivos** en lugar de tablas SQL. Al final de cada módulo exportamos `extracted_<etiqueta>` en los tres formatos más comunes y comparamos.

| Formato | Cómo guarda los datos | Legible por humanos |
|---|---|---|
| **CSV** | Texto, una fila por línea | Sí |
| **JSON** | Texto con pares `"clave": valor` | Sí |
| **Parquet** | Binario, **por columnas**, comprimido y con tipos de dato | No |

**Lectura del resultado:**

- **Tamaño:** Parquet es el más pequeño (comprime cada columna). JSON es el más grande porque repite el nombre de cada columna en cada registro.
- **Tipos:** **CSV** pierde la fecha (`_extracted_at` regresa como texto): solo guarda texto y pandas adivina los tipos al leer. **JSON** no tiene tipo fecha; pandas la reconoce porque la columna termina en `_at`, pero es una suposición. **Parquet** guarda el tipo de cada columna dentro del archivo.

| Si necesitas… | Usa |
|---|---|
| Compartir datos para abrir en Excel | **CSV** |
| Intercambiar datos con una API | **JSON** |
| Guardar tablas grandes para análisis | **Parquet** |

---
# MÓDULO 1 — `vg_engagement`

`raw_vg_engagement` → `extracted_raw_vg_engagement`

Ejecuta las celdas de este módulo **en orden**: la primera define las tablas y el control de cambios que usan todas las demás.

In [ ]:
# ── Módulo 1: tablas y control de cambios de vg_engagement ──────────────────────────

TABLA_RAW       = 'raw_vg_engagement'               # origen
TABLA_EXTRACTED = 'extracted_raw_vg_engagement'     # destino (sin espacios al final: SQLite los toma como parte del nombre)
CDC_PATH        = f'{LOGS_PATH}/cdc_control_{TABLA_EXTRACTED}.json'   # control de cambios de ESTA tabla
COLS_TIPOS      = ['Rating', 'Plays', '_batch_id', '_extracted_at']   # columnas para comparar tipos entre formatos


print()
print(f'Módulo 1  : vg_engagement')
print(f'Origen    : {TABLA_RAW}')
print(f'Destino   : {TABLA_EXTRACTED}')
print(f'CDC       : {CDC_PATH}')
print()

### 1.1 Revisar el origen: `raw_vg_engagement` (solo lectura)

In [ ]:
# ── Leer raw_<etiqueta> en modo solo lectura ─────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

df_raw = pd.read_sql_query(f'SELECT * FROM {TABLA_RAW}', conn)
conn.close()

inspeccionar_dataset(df_raw, nombre=TABLA_RAW)

In [ ]:
# ── Prueba con las primeras 3 filas ───────────────────────────

prueba = df_raw.head(3).copy()
prueba['_row_hash'] = calcular_huella(df_raw.head(3))
prueba


### 1.2 Extracción con control de cambios

In [ ]:
# ── Estado del control de cambios ANTES de extraer ─────────────────────────

print(json.dumps(leer_cdc(), indent=2, ensure_ascii=False))

In [ ]:
# ── Ejecutar ───────────────────────────────────────────────────────────────

df_nuevas = extraer()

In [ ]:
# ── Ver la tabla EXTRACTED ─────────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED} LIMIT 5', conn))

conn.close()

In [ ]:
# ── Estado del control de cambios ──────────────────────────────────────────

print(json.dumps(leer_cdc(), indent=2, ensure_ascii=False))



### 1.3 Verificar con SQL

In [ ]:
# ── ¿Coinciden RAW y EXTRACTED? ────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT (SELECT COUNT(*) FROM {TABLA_RAW})                       AS filas_raw,
           (SELECT COUNT(*) FROM {TABLA_EXTRACTED})                 AS filas_extracted,
           (SELECT COUNT(DISTINCT _row_hash) FROM {TABLA_EXTRACTED}) AS huellas_distintas
''', conn))

conn.close()

- Si `filas_raw` = `filas_extracted`: la extracción está completa.
- Si `huellas_distintas` es **menor** que `filas_raw`: hay **filas exactamente duplicadas** en `raw_<etiqueta>`. No es un error de la extracción; se anota para limpiarlo en TRANSFORMED.

In [ ]:
# ── ¿Cuántas filas llegaron en cada lote? ──────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT _batch_id, _extracted_at, _source_table, COUNT(*) AS filas
    FROM {TABLA_EXTRACTED}
    GROUP BY _batch_id, _extracted_at, _source_table
    ORDER BY _batch_id
''', conn))

conn.close()

In [ ]:
# ── ¿Los metadatos están completos? (no debe haber nulos) ──────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT COUNT(*)                                                 AS filas,
           SUM(CASE WHEN _batch_id     IS NULL THEN 1 ELSE 0 END)   AS nulos_batch,
           SUM(CASE WHEN _extracted_at IS NULL THEN 1 ELSE 0 END)   AS nulos_fecha,
           SUM(CASE WHEN _row_hash     IS NULL THEN 1 ELSE 0 END)   AS nulos_huella
    FROM {TABLA_EXTRACTED}
''', conn))

conn.close()

### 1.4 Exportar: CSV vs. JSON vs. Parquet

In [ ]:
# ── Leer extracted_<etiqueta> ────────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
df_ext = pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED}', conn)
conn.close()


In [ ]:
# SQLite no tiene tipo fecha: _extracted_at llegó como texto. La convertimos a fecha real.
print()
df_ext['_extracted_at'] = pd.to_datetime(df_ext['_extracted_at'])

df_ext.dtypes.to_frame('tipo').T


In [ ]:
# ── Exportar en los tres formatos ──────────────────────────────────────────

ruta_csv     = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.csv'
ruta_json    = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.json'
ruta_parquet = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.parquet'


df_ext.to_csv(ruta_csv, index=False)
df_ext.to_json(ruta_json, orient='records', date_format='iso', force_ascii=False)
df_ext.to_parquet(ruta_parquet, index=False)



pd.DataFrame({
    'formato'  : ['CSV', 'JSON', 'Parquet'],
    'tamaño_MB': [round(os.path.getsize(r) / 1_048_576, 2) for r in (ruta_csv, ruta_json, ruta_parquet)],
}).set_index('formato')



In [ ]:
# ── Leerlos de vuelta: ¿qué tipos de dato se conservan? ────────────────────

desde_csv     = pd.read_csv(ruta_csv)
desde_json    = pd.read_json(ruta_json, orient='records')
desde_parquet = pd.read_parquet(ruta_parquet)

columnas = COLS_TIPOS

pd.DataFrame({
    'original': df_ext[columnas].dtypes.astype(str),
    'CSV'     : desde_csv[columnas].dtypes.astype(str),
    'JSON'    : desde_json[columnas].dtypes.astype(str),
    'Parquet' : desde_parquet[columnas].dtypes.astype(str),
})

### (Opcional) Empezar de cero — Módulo 1

Borra `extracted_raw_vg_engagement` y su control de cambios para volver a hacer la carga completa. **No toca `raw_vg_engagement`.** Quita los `#` para ejecutarla.

In [ ]:
# conn = sqlite3.connect(DB_PATH)
# conn.execute(f'DROP TABLE IF EXISTS {TABLA_EXTRACTED}')
# conn.commit()
# conn.close()
# if os.path.exists(CDC_PATH):
#     os.remove(CDC_PATH)
# print(f'{TABLA_EXTRACTED} y control de cambios reiniciados')

---
# MÓDULO 2 — `vg_global_codificado`

`raw_vg_global_codificado` → `extracted_vg_global_codificado`

Ejecuta las celdas de este módulo **en orden**: la primera define las tablas y el control de cambios que usan todas las demás.

In [ ]:
# ── Módulo 2: tablas y control de cambios de vg_global_codificado ───────────────────

TABLA_RAW       = 'raw_vg_global_codificado'        # origen
TABLA_EXTRACTED = 'extracted_vg_global_codificado'  # destino (sin espacios al final: SQLite los toma como parte del nombre)
CDC_PATH        = f'{LOGS_PATH}/cdc_control_{TABLA_EXTRACTED}.json'   # control de cambios de ESTA tabla
COLS_TIPOS      = ['Platform', 'Global_Sales', '_batch_id', '_extracted_at']   # columnas para comparar tipos entre formatos


print()
print(f'Módulo 2  : vg_global_codificado')
print(f'Origen    : {TABLA_RAW}')
print(f'Destino   : {TABLA_EXTRACTED}')
print(f'CDC       : {CDC_PATH}')
print()

### 2.1 Revisar el origen: `raw_vg_global_codificado` (solo lectura)

In [ ]:
# ── Leer raw_<etiqueta> en modo solo lectura ─────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

df_raw = pd.read_sql_query(f'SELECT * FROM {TABLA_RAW}', conn)
conn.close()

inspeccionar_dataset(df_raw, nombre=TABLA_RAW)

In [ ]:
# ── Prueba con las primeras 3 filas ───────────────────────────

prueba = df_raw.head(3).copy()
prueba['_row_hash'] = calcular_huella(df_raw.head(3))
prueba


### 2.2 Extracción con control de cambios

In [ ]:
# ── Estado del control de cambios ANTES de extraer ─────────────────────────

print(json.dumps(leer_cdc(), indent=2, ensure_ascii=False))

In [ ]:
# ── Ejecutar ───────────────────────────────────────────────────────────────

df_nuevas = extraer()

In [ ]:
# ── Ver la tabla EXTRACTED ─────────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED} LIMIT 5', conn))

conn.close()

In [ ]:
# ── Estado del control de cambios ──────────────────────────────────────────

print(json.dumps(leer_cdc(), indent=2, ensure_ascii=False))



### 2.3 Verificar con SQL

In [ ]:
# ── ¿Coinciden RAW y EXTRACTED? ────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT (SELECT COUNT(*) FROM {TABLA_RAW})                       AS filas_raw,
           (SELECT COUNT(*) FROM {TABLA_EXTRACTED})                 AS filas_extracted,
           (SELECT COUNT(DISTINCT _row_hash) FROM {TABLA_EXTRACTED}) AS huellas_distintas
''', conn))

conn.close()

- Si `filas_raw` = `filas_extracted`: la extracción está completa.
- Si `huellas_distintas` es **menor** que `filas_raw`: hay **filas exactamente duplicadas** en `raw_<etiqueta>`. No es un error de la extracción; se anota para limpiarlo en TRANSFORMED.

In [ ]:
# ── ¿Cuántas filas llegaron en cada lote? ──────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT _batch_id, _extracted_at, _source_table, COUNT(*) AS filas
    FROM {TABLA_EXTRACTED}
    GROUP BY _batch_id, _extracted_at, _source_table
    ORDER BY _batch_id
''', conn))

conn.close()

In [ ]:
# ── ¿Los metadatos están completos? (no debe haber nulos) ──────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT COUNT(*)                                                 AS filas,
           SUM(CASE WHEN _batch_id     IS NULL THEN 1 ELSE 0 END)   AS nulos_batch,
           SUM(CASE WHEN _extracted_at IS NULL THEN 1 ELSE 0 END)   AS nulos_fecha,
           SUM(CASE WHEN _row_hash     IS NULL THEN 1 ELSE 0 END)   AS nulos_huella
    FROM {TABLA_EXTRACTED}
''', conn))

conn.close()

### 2.4 Exportar: CSV vs. JSON vs. Parquet

In [ ]:
# ── Leer extracted_<etiqueta> ────────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
df_ext = pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED}', conn)
conn.close()


In [ ]:
# SQLite no tiene tipo fecha: _extracted_at llegó como texto. La convertimos a fecha real.
print()
df_ext['_extracted_at'] = pd.to_datetime(df_ext['_extracted_at'])

df_ext.dtypes.to_frame('tipo').T


In [ ]:
# ── Exportar en los tres formatos ──────────────────────────────────────────

ruta_csv     = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.csv'
ruta_json    = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.json'
ruta_parquet = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.parquet'


df_ext.to_csv(ruta_csv, index=False)
df_ext.to_json(ruta_json, orient='records', date_format='iso', force_ascii=False)
df_ext.to_parquet(ruta_parquet, index=False)



pd.DataFrame({
    'formato'  : ['CSV', 'JSON', 'Parquet'],
    'tamaño_MB': [round(os.path.getsize(r) / 1_048_576, 2) for r in (ruta_csv, ruta_json, ruta_parquet)],
}).set_index('formato')



In [ ]:
# ── Leerlos de vuelta: ¿qué tipos de dato se conservan? ────────────────────

desde_csv     = pd.read_csv(ruta_csv)
desde_json    = pd.read_json(ruta_json, orient='records')
desde_parquet = pd.read_parquet(ruta_parquet)

columnas = COLS_TIPOS

pd.DataFrame({
    'original': df_ext[columnas].dtypes.astype(str),
    'CSV'     : desde_csv[columnas].dtypes.astype(str),
    'JSON'    : desde_json[columnas].dtypes.astype(str),
    'Parquet' : desde_parquet[columnas].dtypes.astype(str),
})

### (Opcional) Empezar de cero — Módulo 2

Borra `extracted_vg_global_codificado` y su control de cambios para volver a hacer la carga completa. **No toca `raw_vg_global_codificado`.** Quita los `#` para ejecutarla.

In [ ]:
# conn = sqlite3.connect(DB_PATH)
# conn.execute(f'DROP TABLE IF EXISTS {TABLA_EXTRACTED}')
# conn.commit()
# conn.close()
# if os.path.exists(CDC_PATH):
#     os.remove(CDC_PATH)
# print(f'{TABLA_EXTRACTED} y control de cambios reiniciados')

---
# MÓDULO 3 — `vg_ventas_ratings`

`raw_vg_ventas_ratings` → `extracted_vg_ventas_ratings`

Ejecuta las celdas de este módulo **en orden**: la primera define las tablas y el control de cambios que usan todas las demás.

In [ ]:
# ── Módulo 3: tablas y control de cambios de vg_ventas_ratings ──────────────────────

TABLA_RAW       = 'raw_vg_ventas_ratings'           # origen
TABLA_EXTRACTED = 'extracted_vg_ventas_ratings'     # destino (sin espacios al final: SQLite los toma como parte del nombre)
CDC_PATH        = f'{LOGS_PATH}/cdc_control_{TABLA_EXTRACTED}.json'   # control de cambios de ESTA tabla
COLS_TIPOS      = ['Year_of_Release', 'Global_Sales', '_batch_id', '_extracted_at']   # columnas para comparar tipos entre formatos


print()
print(f'Módulo 3  : vg_ventas_ratings')
print(f'Origen    : {TABLA_RAW}')
print(f'Destino   : {TABLA_EXTRACTED}')
print(f'CDC       : {CDC_PATH}')
print()

### 3.1 Revisar el origen: `raw_vg_ventas_ratings` (solo lectura)

In [ ]:
# ── Leer raw_<etiqueta> en modo solo lectura ─────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

df_raw = pd.read_sql_query(f'SELECT * FROM {TABLA_RAW}', conn)
conn.close()

inspeccionar_dataset(df_raw, nombre=TABLA_RAW)

In [ ]:
# ── Prueba con las primeras 3 filas ───────────────────────────

prueba = df_raw.head(3).copy()
prueba['_row_hash'] = calcular_huella(df_raw.head(3))
prueba


### 3.2 Extracción con control de cambios

In [ ]:
# ── Estado del control de cambios ANTES de extraer ─────────────────────────

print(json.dumps(leer_cdc(), indent=2, ensure_ascii=False))

In [ ]:
# ── Ejecutar ───────────────────────────────────────────────────────────────

df_nuevas = extraer()

In [ ]:
# ── Ver la tabla EXTRACTED ─────────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED} LIMIT 5', conn))

conn.close()

In [ ]:
# ── Estado del control de cambios ──────────────────────────────────────────

print(json.dumps(leer_cdc(), indent=2, ensure_ascii=False))



### 3.3 Verificar con SQL

In [ ]:
# ── ¿Coinciden RAW y EXTRACTED? ────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT (SELECT COUNT(*) FROM {TABLA_RAW})                       AS filas_raw,
           (SELECT COUNT(*) FROM {TABLA_EXTRACTED})                 AS filas_extracted,
           (SELECT COUNT(DISTINCT _row_hash) FROM {TABLA_EXTRACTED}) AS huellas_distintas
''', conn))

conn.close()

- Si `filas_raw` = `filas_extracted`: la extracción está completa.
- Si `huellas_distintas` es **menor** que `filas_raw`: hay **filas exactamente duplicadas** en `raw_<etiqueta>`. No es un error de la extracción; se anota para limpiarlo en TRANSFORMED.

In [ ]:
# ── ¿Cuántas filas llegaron en cada lote? ──────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT _batch_id, _extracted_at, _source_table, COUNT(*) AS filas
    FROM {TABLA_EXTRACTED}
    GROUP BY _batch_id, _extracted_at, _source_table
    ORDER BY _batch_id
''', conn))

conn.close()

In [ ]:
# ── ¿Los metadatos están completos? (no debe haber nulos) ──────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT COUNT(*)                                                 AS filas,
           SUM(CASE WHEN _batch_id     IS NULL THEN 1 ELSE 0 END)   AS nulos_batch,
           SUM(CASE WHEN _extracted_at IS NULL THEN 1 ELSE 0 END)   AS nulos_fecha,
           SUM(CASE WHEN _row_hash     IS NULL THEN 1 ELSE 0 END)   AS nulos_huella
    FROM {TABLA_EXTRACTED}
''', conn))

conn.close()

### 3.4 Exportar: CSV vs. JSON vs. Parquet

In [ ]:
# ── Leer extracted_<etiqueta> ────────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
df_ext = pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED}', conn)
conn.close()


In [ ]:
# SQLite no tiene tipo fecha: _extracted_at llegó como texto. La convertimos a fecha real.
print()
df_ext['_extracted_at'] = pd.to_datetime(df_ext['_extracted_at'])

df_ext.dtypes.to_frame('tipo').T


In [ ]:
# ── Exportar en los tres formatos ──────────────────────────────────────────

ruta_csv     = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.csv'
ruta_json    = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.json'
ruta_parquet = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.parquet'


df_ext.to_csv(ruta_csv, index=False)
df_ext.to_json(ruta_json, orient='records', date_format='iso', force_ascii=False)
df_ext.to_parquet(ruta_parquet, index=False)



pd.DataFrame({
    'formato'  : ['CSV', 'JSON', 'Parquet'],
    'tamaño_MB': [round(os.path.getsize(r) / 1_048_576, 2) for r in (ruta_csv, ruta_json, ruta_parquet)],
}).set_index('formato')



In [ ]:
# ── Leerlos de vuelta: ¿qué tipos de dato se conservan? ────────────────────

desde_csv     = pd.read_csv(ruta_csv)
desde_json    = pd.read_json(ruta_json, orient='records')
desde_parquet = pd.read_parquet(ruta_parquet)

columnas = COLS_TIPOS

pd.DataFrame({
    'original': df_ext[columnas].dtypes.astype(str),
    'CSV'     : desde_csv[columnas].dtypes.astype(str),
    'JSON'    : desde_json[columnas].dtypes.astype(str),
    'Parquet' : desde_parquet[columnas].dtypes.astype(str),
})

### (Opcional) Empezar de cero — Módulo 3

Borra `extracted_vg_ventas_ratings` y su control de cambios para volver a hacer la carga completa. **No toca `raw_vg_ventas_ratings`.** Quita los `#` para ejecutarla.

In [ ]:
# conn = sqlite3.connect(DB_PATH)
# conn.execute(f'DROP TABLE IF EXISTS {TABLA_EXTRACTED}')
# conn.commit()
# conn.close()
# if os.path.exists(CDC_PATH):
#     os.remove(CDC_PATH)
# print(f'{TABLA_EXTRACTED} y control de cambios reiniciados')

---
## CIERRE — Log de operaciones de los 3 módulos

El log es **uno solo** para las 3 tablas: cada línea indica su tabla de origen y destino.

In [ ]:
# ── Log de operaciones ─────────────────────────────────────────────────────

ver_log()



In [ ]:
# ── Así se ve el archivo del log por dentro ────

with open(LOG_PATH, encoding='utf-8') as f:
    print(f.read())


---
### Resumen

| Paso | Qué hicimos |
|---|---|
| Parte común | Definimos **una vez** las funciones de huella, control de cambios, log y `extraer()` |
| Módulos 1–3 | Para cada dataset: leímos `raw_<etiqueta>` en **solo lectura**, agregamos los **metadatos** (`_batch_id`, `_extracted_at`, `_source_table`, `_row_hash`) y extrajimos solo las filas nuevas (`append`) |
| Control de cambios | Cada tabla registra su estado en **su propio** `cdc_control_<tabla>.json` |
| Verificación | En cada módulo, **SQL**: conteos RAW vs. EXTRACTED, lotes y metadatos completos |
| Formatos | En cada módulo comparamos **CSV, JSON y Parquet** |
| Cierre | Un **log** compartido con todas las operaciones |

**Siguiente notebook:** TRANSFORMED, que leerá **`extracted_<etiqueta>`** (nunca `raw_<etiqueta>`) para limpiar los datos.